<a href="https://colab.research.google.com/github/minishiftvg/osint-agent/blob/main/finetuning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Fine-Tuning di un LLM Leggero con QLoRA su Google Colab

## Panoramica del Progetto

Questo notebook implementa il fine-tuning di **Phi-3.5-mini-instruct** (Microsoft, 3.8B parametri) usando la tecnica **QLoRA** (Quantized Low-Rank Adaptation).

### Perché Phi-3.5-mini?
- ✅ Completamente **gratuito** (licenza MIT)
- ✅ **3.8B parametri** — gira su GPU T4 gratuita di Colab
- ✅ Ottimo rapporto qualità/dimensioni
- ✅ Già ottimizzato per istruzioni (instruct-tuned)

### Perché QLoRA?
- ✅ Addestra solo ~1% dei parametri (adattatori LoRA)
- ✅ Il modello base viene quantizzato a **4-bit** (meno RAM/VRAM)
- ✅ Risultati quasi uguali al full fine-tuning
- ✅ Perfetto per GPU consumer (T4, A100 gratuiti)

### Architettura del flusso:
```
Documenti (PDF/TXT/DOCX) → Estrazione testo → Creazione dataset Q&A → Fine-tuning QLoRA → Modello personalizzato
```

---
⚠️ **IMPORTANTE**: Prima di iniziare, vai su `Runtime → Cambia tipo di runtime → GPU (T4)`

## 📦 Cella 1: Installazione delle Dipendenze

Installiamo tutte le librerie necessarie:
- **transformers**: libreria HuggingFace per caricare e usare LLM
- **peft**: Parameter-Efficient Fine-Tuning (implementa LoRA)
- **trl**: Transformer Reinforcement Learning (SFTTrainer per il training supervisionato)
- **bitsandbytes**: quantizzazione a 4-bit (cuore di QLoRA)
- **accelerate**: gestione automatica di GPU/CPU
- **datasets**: gestione dataset HuggingFace
- **PyMuPDF (fitz)**: lettura PDF
- **python-docx**: lettura file Word (.docx)

In [1]:
# @title
# ============================================================
# CELLA 1: INSTALLAZIONE DIPENDENZE
# ============================================================
# Installiamo tutto ciò che serve per il fine-tuning
# Il flag -q sopprime l'output verboso di pip

# ============================================================
# CELLA 1 (CORRETTA): INSTALLAZIONE DIPENDENZE
# ============================================================
# Prima di tutto, disinstalla le versioni problematiche
!pip uninstall -y bitsandbytes triton 2>/dev/null

# Installa bitsandbytes dalla versione precompilata per CUDA su Colab
# La versione 0.45.x ha risolto i problemi con triton.ops
!pip install -q bitsandbytes==0.45.3

# Installa il resto delle dipendenze
!pip install -q transformers==4.44.2
!pip install -q peft==0.12.0
!pip install -q trl==0.11.1
!pip install -q accelerate==0.34.2
!pip install -q datasets==2.21.0
!pip install -q PyMuPDF
!pip install -q python-docx
!pip install -q sentencepiece protobuf

print("✅ Installazione completata!")
print("✅ Tutte le dipendenze installate correttamente!")

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
# CELLA DI VERIFICA (esegui dopo il restart)
import subprocess, sys

# CELLA DI VERIFICA CORRETTA per bitsandbytes 0.45.x
import torch
print(f"CUDA disponibile: {torch.cuda.is_available()}")
print(f"Versione CUDA:    {torch.version.cuda}")
print(f"GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NESSUNA'}")

import bitsandbytes as bnb
print(f"\n✅ bitsandbytes versione: {bnb.__version__}")

# In 0.45.x il check si fa così:
from bitsandbytes.functional import quantize_blockwise
test_tensor = torch.randn(64, 64).cuda()
quantized, state = quantize_blockwise(test_tensor)
print(f"✅ Quantizzazione 4-bit funzionante su GPU!")
print(f"\n🎉 Tutto OK! Procedi con le celle successive.")

CUDA disponibile: True
Versione CUDA:    12.8
GPU:              Tesla T4

✅ bitsandbytes versione: 0.45.3
✅ Quantizzazione 4-bit funzionante su GPU!

🎉 Tutto OK! Procedi con le celle successive.


In [2]:
# ============================================================
# Installazione dipendenza mancante: Triton
# ============================================================
# BitsAndBytes richiede Triton per alcune delle sue operazioni CUDA.
# Se si verifica un 'ModuleNotFoundError: No module named 'triton.ops'',
# è necessario installare Triton.

!pip install -q triton

print("✅ Triton installato correttamente!")

✅ Triton installato correttamente!


## ⚙️ Cella 2: Import e Configurazione GPU

Carichiamo tutte le librerie e verifichiamo che la GPU sia disponibile.

In [3]:
# @title
# ============================================================
# CELLA 2: IMPORT LIBRERIE E VERIFICA GPU
# ============================================================

import os
import json
import re
import torch
import fitz                          # PyMuPDF per leggere PDF
from docx import Document            # python-docx per leggere Word
from pathlib import Path
from typing import List, Dict

# HuggingFace Transformers
from transformers import (
    AutoTokenizer,                   # Carica il tokenizer del modello
    AutoModelForCausalLM,            # Carica il modello language model
    BitsAndBytesConfig,              # Configurazione quantizzazione 4-bit
    TrainingArguments,               # Iperparametri di training
    pipeline                         # Pipeline per inference rapida
)

# PEFT - Parameter Efficient Fine-Tuning
from peft import (
    LoraConfig,                      # Configurazione adattatori LoRA
    get_peft_model,                  # Applica LoRA al modello
    TaskType                         # Tipo di task (generazione causale)
)

# TRL - Training con istruzioni
from trl import SFTTrainer           # Supervised Fine-Tuning Trainer

# Dataset HuggingFace
from datasets import Dataset

# ---- Verifica GPU ----
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU trovata: {gpu_name}")
    print(f"   Memoria VRAM disponibile: {gpu_mem:.1f} GB")
else:
    print("❌ GPU NON trovata! Vai su Runtime → Cambia tipo di runtime → GPU")
    raise SystemExit("GPU richiesta per il fine-tuning")

# Imposta il device principale
DEVICE = "cuda"
print(f"\n🔧 Device attivo: {DEVICE}")

✅ GPU trovata: Tesla T4
   Memoria VRAM disponibile: 15.6 GB

🔧 Device attivo: cuda


## 📁 Cella 3: Upload dei Documenti

Carica i tuoi file (PDF, TXT, DOCX) direttamente su Colab.
Puoi caricare uno o più documenti che useranno come 'conoscenza' per addestrare il modello.

In [4]:
# @title
# ============================================================
# CELLA 3: UPLOAD DEI DOCUMENTI
# ============================================================
# Usiamo la funzione files.upload() di Google Colab
# che apre un dialogo per selezionare i file dal tuo computer

from google.colab import files

print("📂 Seleziona i tuoi documenti (PDF, TXT, DOCX)...")
print("   Puoi caricare più file contemporaneamente!\n")

# files.upload() restituisce un dizionario {nome_file: contenuto_bytes}
uploaded = files.upload()

# Salviamo i file in una cartella dedicata
DOCS_DIR = "/content/documents"
os.makedirs(DOCS_DIR, exist_ok=True)

for filename, content in uploaded.items():
    # Salva ogni file caricato nella cartella documents
    filepath = os.path.join(DOCS_DIR, filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"✅ Salvato: {filename} ({len(content)/1024:.1f} KB)")

print(f"\n📁 Tutti i documenti sono in: {DOCS_DIR}")

📂 Seleziona i tuoi documenti (PDF, TXT, DOCX)...
   Puoi caricare più file contemporaneamente!



Saving Aleph.docx to Aleph.docx
Saving CMDB.docx to CMDB.docx
Saving Configurazione Dual NIC su Red Hat Linux 9.4.docx to Configurazione Dual NIC su Red Hat Linux 9.4.docx
Saving Confluent.docx to Confluent.docx
Saving CreazioneCertificatoSSL.docx to CreazioneCertificatoSSL.docx
Saving CYBERSECURITY.docx to CYBERSECURITY.docx
Saving FortiGate-Grok.docx to FortiGate-Grok.docx
Saving Fortinet-Firewall.docx to Fortinet-Firewall.docx
Saving Glossario AI.docx to Glossario AI.docx
Saving gpu.docx to gpu.docx
Saving guida_dna_center.docx to guida_dna_center.docx
Saving guida_tenable.docx to guida_tenable.docx
Saving Infodas-DS.docx to Infodas-DS.docx
✅ Salvato: Aleph.docx (467.1 KB)
✅ Salvato: CMDB.docx (338.1 KB)
✅ Salvato: Configurazione Dual NIC su Red Hat Linux 9.4.docx (30.6 KB)
✅ Salvato: Confluent.docx (143.7 KB)
✅ Salvato: CreazioneCertificatoSSL.docx (76.7 KB)
✅ Salvato: CYBERSECURITY.docx (12601.4 KB)
✅ Salvato: FortiGate-Grok.docx (1776.4 KB)
✅ Salvato: Fortinet-Firewall.docx (117.

## 📄 Cella 4: Estrattore di Testo Multi-Formato

Definiamo le funzioni per estrarre il testo grezzo da PDF, TXT e DOCX.

In [5]:
# @title
# ============================================================
# CELLA 4: FUNZIONI DI ESTRAZIONE TESTO
# ============================================================

def extract_text_from_pdf(filepath: str) -> str:
    """
    Estrae tutto il testo da un file PDF.

    Usa PyMuPDF (fitz) che è molto robusto:
    - Gestisce PDF con immagini, tabelle, colonne multiple
    - Mantiene l'ordine di lettura corretto
    - Molto più affidabile di pypdf o pdfplumber per testi complessi

    Args:
        filepath: percorso al file PDF
    Returns:
        Testo estratto come stringa unica
    """
    text_parts = []

    # Apre il documento PDF
    doc = fitz.open(filepath)

    print(f"   PDF: {Path(filepath).name} — {len(doc)} pagine")

    for page_num, page in enumerate(doc):
        # get_text("text") estrae il testo plain text
        # Altre opzioni: "html", "dict", "blocks" per formati strutturati
        page_text = page.get_text("text")
        if page_text.strip():  # Ignora pagine vuote
            text_parts.append(page_text)

    doc.close()
    return "\n".join(text_parts)


def extract_text_from_docx(filepath: str) -> str:
    """
    Estrae tutto il testo da un file Word (.docx).

    python-docx legge i paragrafi del documento XML interno.
    Nota: non gestisce le immagini embedded (solo testo).

    Args:
        filepath: percorso al file .docx
    Returns:
        Testo estratto come stringa unica
    """
    doc = Document(filepath)

    # .paragraphs è una lista di oggetti Paragraph
    # Ogni paragrafo ha l'attributo .text con il testo plain
    text_parts = [para.text for para in doc.paragraphs if para.text.strip()]

    print(f"   DOCX: {Path(filepath).name} — {len(doc.paragraphs)} paragrafi")
    return "\n".join(text_parts)


def extract_text_from_txt(filepath: str) -> str:
    """
    Legge un file di testo plain.

    Prova prima UTF-8 (standard), poi latin-1 come fallback
    per gestire file con encoding non standard (es. file Windows).

    Args:
        filepath: percorso al file .txt
    Returns:
        Contenuto del file come stringa
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    except UnicodeDecodeError:
        # Fallback per file Windows con encoding ANSI
        with open(filepath, 'r', encoding='latin-1') as f:
            text = f.read()

    print(f"   TXT: {Path(filepath).name} — {len(text)} caratteri")
    return text


def extract_all_documents(docs_dir: str) -> Dict[str, str]:
    """
    Funzione principale che scansiona una cartella e estrae
    il testo da tutti i file supportati (PDF, TXT, DOCX).

    Args:
        docs_dir: cartella contenente i documenti
    Returns:
        Dizionario {nome_file: testo_estratto}
    """
    documents = {}

    # Mappa estensione → funzione estrattore
    extractors = {
        '.pdf':  extract_text_from_pdf,
        '.txt':  extract_text_from_txt,
        '.docx': extract_text_from_docx,
        '.doc':  extract_text_from_docx,  # Stesso estrattore per .doc
    }

    print("📄 Estrazione testo dai documenti...")

    for filename in sorted(os.listdir(docs_dir)):
        filepath = os.path.join(docs_dir, filename)
        ext = Path(filename).suffix.lower()

        if ext in extractors:
            try:
                text = extractors[ext](filepath)
                if text.strip():
                    documents[filename] = text
                    print(f"   ✅ Estratti {len(text):,} caratteri")
                else:
                    print(f"   ⚠️  {filename}: documento vuoto, saltato")
            except Exception as e:
                print(f"   ❌ Errore con {filename}: {e}")
        else:
            print(f"   ℹ️  {filename}: formato non supportato, saltato")

    total_chars = sum(len(t) for t in documents.values())
    print(f"\n📊 Totale: {len(documents)} documenti, {total_chars:,} caratteri")
    return documents


# Esegui l'estrazione su tutti i documenti caricati
documents = extract_all_documents(DOCS_DIR)

📄 Estrazione testo dai documenti...
   DOCX: Aleph.docx — 4871 paragrafi
   ✅ Estratti 170,762 caratteri
   DOCX: CMDB.docx — 4851 paragrafi
   ✅ Estratti 247,319 caratteri
   DOCX: CYBERSECURITY.docx — 4515 paragrafi
   ✅ Estratti 333,406 caratteri
   DOCX: Configurazione Dual NIC su Red Hat Linux 9.4.docx — 261 paragrafi
   ✅ Estratti 11,173 caratteri
   DOCX: Confluent.docx — 1173 paragrafi
   ✅ Estratti 99,279 caratteri
   DOCX: CreazioneCertificatoSSL.docx — 1003 paragrafi
   ✅ Estratti 39,165 caratteri
   DOCX: FortiGate-Grok.docx — 1024 paragrafi
   ✅ Estratti 50,294 caratteri
   DOCX: Fortinet-Firewall.docx — 1415 paragrafi
   ✅ Estratti 86,185 caratteri
   DOCX: Glossario AI.docx — 350 paragrafi
   ✅ Estratti 43,639 caratteri
   DOCX: Infodas-DS.docx — 428 paragrafi
   ✅ Estratti 51,518 caratteri
   DOCX: gpu.docx — 1148 paragrafi
   ✅ Estratti 54,847 caratteri
   DOCX: guida_dna_center.docx — 505 paragrafi
   ✅ Estratti 35,407 caratteri
   DOCX: guida_tenable.docx — 329 parag

## 🔨 Cella 5: Preprocessing e Chunking del Testo

Il testo grezzo va pulito e suddiviso in **chunk** (porzioni) di dimensione gestibile.

**Perché i chunk?** I modelli hanno una finestra di contesto limitata (es. 2048 token). Dobbiamo spezzare testi lunghi in pezzi che stiano dentro questa finestra.

In [6]:
# @title
# ============================================================
# CELLA 5: PREPROCESSING E CHUNKING
# ============================================================

def clean_text(text: str) -> str:
    """
    Pulisce il testo grezzo rimuovendo artefatti comuni.

    Operazioni:
    - Rimuove righe eccessivamente corte (numeri di pagina, header)
    - Normalizza spazi multipli e ritorni a capo
    - Rimuove caratteri speciali non printable
    """
    # Rimuovi caratteri non stampabili (es. \x00, \x01 dai PDF)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Normalizza spazi multipli su una singola riga
    text = re.sub(r' +', ' ', text)

    # Riduci linee vuote multiple a massimo 2
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Filtra righe molto corte (probabilmente numeri di pagina o rumori)
    lines = text.split('\n')
    filtered_lines = [
        line for line in lines
        if len(line.strip()) > 3  # Tieni solo righe con più di 3 caratteri
        or line.strip() == ''     # Mantieni righe vuote per separazione
    ]

    return '\n'.join(filtered_lines).strip()


def chunk_text(text: str, chunk_size: int = 512, overlap: int = 64) -> List[str]:
    """
    Divide il testo in chunk sovrapposti.

    Parametri chiave:
    - chunk_size: dimensione di ogni chunk in PAROLE (non caratteri)
      → 512 parole ≈ 700-800 token, adatto per contesti di 2048 token
    - overlap: quante parole condividono chunk adiacenti
      → L'overlap evita di troncare concetti a metà tra due chunk
      → Es: chunk1=[0:512], chunk2=[448:960] con overlap=64

    Divide per PAROLE invece che caratteri perché:
    - Le parole non vengono spezzate a metà
    - Meglio correlato al numero di token
    """
    words = text.split()  # Dividi il testo in lista di parole
    chunks = []

    # Scorri il testo con passo (chunk_size - overlap)
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        # Prendi le parole da i a i+chunk_size
        chunk_words = words[i : i + chunk_size]

        if len(chunk_words) < 20:  # Ignora chunk troppo corti (< 20 parole)
            continue

        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text)

    return chunks


def process_documents(documents: Dict[str, str],
                      chunk_size: int = 512,
                      overlap: int = 64) -> List[str]:
    """
    Pipeline completa: pulisce i documenti e li divide in chunk.

    Restituisce una lista flat di tutti i chunk da tutti i documenti.
    Questi chunk sono il materiale grezzo da cui costruiremo il dataset.
    """
    all_chunks = []

    for filename, text in documents.items():
        print(f"\n🔄 Processing: {filename}")

        # Step 1: Pulisci il testo
        cleaned = clean_text(text)
        print(f"   Dopo pulizia: {len(cleaned):,} caratteri")

        # Step 2: Dividi in chunk
        chunks = chunk_text(cleaned, chunk_size=chunk_size, overlap=overlap)
        print(f"   Chunk creati: {len(chunks)}")

        # Aggiungi la fonte come prefisso di ogni chunk
        # (utile per debug e per dare contesto al modello)
        tagged_chunks = [
            f"[Fonte: {filename}]\n{chunk}"
            for chunk in chunks
        ]
        all_chunks.extend(tagged_chunks)

    print(f"\n✅ Totale chunk pronti per il training: {len(all_chunks)}")
    return all_chunks


# Esegui il processing
# chunk_size=512 parole ≈ 680 token (sicuro per context window di 2048)
# overlap=64 parole = circa 85 token di sovrapposizione
text_chunks = process_documents(documents, chunk_size=512, overlap=64)


🔄 Processing: Aleph.docx
   Dopo pulizia: 164,563 caratteri
   Chunk creati: 48

🔄 Processing: CMDB.docx
   Dopo pulizia: 246,305 caratteri
   Chunk creati: 83

🔄 Processing: CYBERSECURITY.docx
   Dopo pulizia: 330,170 caratteri
   Chunk creati: 108

🔄 Processing: Configurazione Dual NIC su Red Hat Linux 9.4.docx
   Dopo pulizia: 11,158 caratteri
   Chunk creati: 4

🔄 Processing: Confluent.docx
   Dopo pulizia: 98,362 caratteri
   Chunk creati: 32

🔄 Processing: CreazioneCertificatoSSL.docx
   Dopo pulizia: 38,765 caratteri
   Chunk creati: 12

🔄 Processing: FortiGate-Grok.docx
   Dopo pulizia: 49,660 caratteri
   Chunk creati: 17

🔄 Processing: Fortinet-Firewall.docx
   Dopo pulizia: 85,482 caratteri
   Chunk creati: 30

🔄 Processing: Glossario AI.docx
   Dopo pulizia: 43,521 caratteri
   Chunk creati: 14

🔄 Processing: Infodas-DS.docx
   Dopo pulizia: 51,518 caratteri
   Chunk creati: 17

🔄 Processing: gpu.docx
   Dopo pulizia: 51,545 caratteri
   Chunk creati: 15

🔄 Processing: gui

## 🗂️ Cella 6: Creazione del Dataset di Training

Per il fine-tuning **instruction-following**, trasformiamo i chunk di testo in coppie **istruzione → risposta**. Il modello imparerà a rispondere a domande basandosi sul contenuto dei documenti.

Il formato scelto è lo stile **ChatML** (usato da Phi-3):
```
<|user|>\nDomanda<|end|>\n<|assistant|>\nRisposta<|end|>

```
Questa frase descrive un passaggio fondamentale nel mondo della GenAI: il momento in cui prendi del testo grezzo (i tuoi documenti) e lo trasformi nel carburante adatto per insegnare a un modello linguistico come comportarsi da assistente virtuale, nello specifico usando la sintassi richiesta dalla famiglia di modelli Phi-3 di Microsoft.Vediamo nel dettaglio cosa significa ogni singolo concetto.

1. Fine-tuning "Instruction-Following"Un modello LLM "base" (nato solo dal pre-training) è fondamentalmente un potentissimo completatore di testo. Se gli scrivi "Qual è la capitale della Francia?", potrebbe rispondere con "E qual è la capitale della Germania?" perché ha visto molti quiz online durante il suo addestramento.Il Fine-tuning Instruction-Following è il processo di addestramento aggiuntivo che "educa" il modello a seguire i comandi. Gli insegna che quando l'utente fa una domanda (Istruzione), il suo compito non è continuare a scrivere domande, ma fornire la soluzione (Risposta).

2. Trasformare i "chunk" in coppie Istruzione $\rightarrow$ RispostaPrima di questa cella, il tuo codice ha probabilmente preso dei documenti lunghi (PDF, manuali, file di testo) e li ha fatti a pezzetti (chunk).Tuttavia, non puoi dare in pasto al modello un pezzo di testo piatto se vuoi che impari a dialogare. Questa cella si occupa di:Prendere un chunk (es. "La terra impiega 365 giorni a fare un giro intorno al sole").Generare (spesso usando un LLM più grande o un template) una domanda basata su quel chunk ("Quanto dura l'anno terrestre?").Associare la risposta corretta estratta dal testo ("La terra impiega 365 giorni...").In questo modo crei il Dataset di Training: una collezione di centinaia o migliaia di esempi "Domanda-Risposta" basati sui tuoi dati personalizzati.

3. Il formato ChatML (e lo stile Phi-3)I modelli di intelligenza artificiale non capiscono nativamente chi sta parlando; vedono solo un unico lungo flusso di testo. Per fargli capire quando parla l'utente e quando tocca a loro, si usano dei token speciali di delimitazione.Il formato ChatML (Chat Markup Language) organizza il testo in blocchi espliciti. Nel caso specifico di Phi-3, la struttura rigida che il modello si aspetta di vedere durante l'addestramento è questa:<|user|>\n: È il tag di apertura che dice al modello: "Attenzione, da qui in poi c'è la richiesta dell'essere umano".Domanda: Il testo effettivo della domanda.<|end|>\n: Il tag di chiusura che dice: "L'utente ha finito di parlare".<|assistant|>\n: Il tag che dice al modello: "Da qui in poi c'è la risposta ideale che ti devi abituare a dare".Risposta: Il testo della risposta corretta.<|end|>: Il tag finale che chiude il blocco della risposta.x


Un esempio pratico
Se il tuo script elabora un documento aziendale sulle policy delle ferie, il codice della "Cella 6" prenderà i dati e genererà una stringa di testo formattata esattamente così da passare all'algoritmo di training:Plaintext<|user|>
Quanti giorni di preavviso servono per chiedere le ferie estive?<|end|>
<|assistant|>
In base al regolamento aziendale, le ferie estive devono essere richieste con almeno 15 giorni di preavviso.<|end|>
Perché è fondamentale farlo?Se addestrassi il modello formattando il testo in modo diverso (ad esempio usando U: e A:), il modello Phi-3 non capirebbe i confini della conversazione, perché i suoi circuiti interni sono stati progettati e "pre-addestrati" da Microsoft per riconoscere specificamente i token <|user|> e <|assistant|>. Mantenere questo standard garantisce che il fine-tuning abbia successo e che il modello impari a rispondere sfruttando al massimo la sua logica nativa.


In [7]:
# ============================================================
# CELLA 6: CREAZIONE DATASET DI TRAINING
# ============================================================

def create_training_examples(chunks: List[str],
                              system_prompt: str = None) -> List[Dict]:
    """
    Converte i chunk di testo in esempi di training nel formato
    istruzione-risposta (Instruction Following).

    STRATEGIA:
    Per ogni chunk creiamo 3 tipi di esempi:
    1. 'Riassumi questo testo' → il modello impara a sintetizzare
    2. 'Analizza questo contenuto' → comprensione profonda
    3. 'Spiega i concetti chiave' → estrazione di conoscenza

    Per produzione, si consiglia di aggiungere vere coppie Q&A
    create manualmente o con un LLM più grande (es. GPT-4).

    Args:
        chunks: lista di chunk di testo
        system_prompt: istruzione di sistema che definisce il comportamento
    Returns:
        Lista di dict {text: "testo formattato per training"}
    """

    if system_prompt is None:
        system_prompt = (
            "Sei un assistente esperto. Rispondi in modo accurato e dettagliato "
            "basandoti sulle informazioni fornite."
        )

    # Template di istruzioni diverse per variare il dataset
    # Più varietà = modello più robusto e generalizzabile
    instruction_templates = [
        "Leggi attentamente il seguente testo e fornisci un riassunto dettagliato:\n\n{chunk}",
        "Analizza il seguente contenuto e identifica i punti principali:\n\n{chunk}",
        "Basandoti sul seguente testo, spiega i concetti chiave in modo chiaro:\n\n{chunk}",
        "Esamina il seguente brano e fornisci una spiegazione approfondita:\n\n{chunk}",
        "Che cosa descrive o spiega il seguente testo? Analizzalo nel dettaglio:\n\n{chunk}",
    ]

    # Template risposta: invitiamo il modello a rispondere professionalmente
    response_template = (
        "Basandomi sul testo fornito, ecco l'analisi:\n\n{chunk}\n\n"
        "Il testo illustra questi elementi fondamentali che ho elaborato e sintetizzato sopra."
    )

    examples = []

    for i, chunk in enumerate(chunks):
        # Usa un template diverso per ogni chunk (ciclicamente)
        # % len(templates) garantisce che non si vada fuori indice
        template_idx = i % len(instruction_templates)
        instruction = instruction_templates[template_idx].format(chunk=chunk)
        response    = response_template.format(chunk=chunk)

        # ---- Formato ChatML per Phi-3 ----
        # Phi-3 usa questo formato speciale con token <|...|>
        # Il system prompt definisce il ruolo/persona del modello
        # IMPORTANTE: il formato deve essere ESATTAMENTE questo
        # altrimenti il tokenizer non lo riconosce correttamente
        formatted_text = (
            f"<|system|>\n{system_prompt}<|end|>\n"
            f"<|user|>\n{instruction}<|end|>\n"
            f"<|assistant|>\n{response}<|end|>"
        )

        examples.append({"text": formatted_text})

    return examples


# ---- Crea gli esempi di training ----
# Personalizza il system_prompt con la persona del tuo assistente!
SYSTEM_PROMPT = (
    "Sei un assistente AI specializzato nell'analisi dei documenti forniti. "
    "Rispondi sempre in italiano in modo preciso, dettagliato e professionale. "
    "Le tue risposte sono basate esclusivamente sulle informazioni contenute nei documenti."
)

training_examples = create_training_examples(text_chunks, system_prompt=SYSTEM_PROMPT)

print(f"✅ Creati {len(training_examples)} esempi di training")
print(f"\n📋 Esempio di training (primo elemento):")
print("-" * 60)
print(training_examples[0]['text'][:800] + "...")
print("-" * 60)

✅ Creati 401 esempi di training

📋 Esempio di training (primo elemento):
------------------------------------------------------------
<|system|>
Sei un assistente AI specializzato nell'analisi dei documenti forniti. Rispondi sempre in italiano in modo preciso, dettagliato e professionale. Le tue risposte sono basate esclusivamente sulle informazioni contenute nei documenti.<|end|>
<|user|>
Leggi attentamente il seguente testo e fornisci un riassunto dettagliato:

[Fonte: Aleph.docx]
Stai creando quello che in gergo tecnico si chiama uno "stack a matrioska": hai l'hypervisor ESXi alla base, sopra una VM che esegue OKD (la distribuzione community di OpenShift), e dentro OKD eseguirai Aleph in container orchestrati da Kubernetes. Ogni livello aggiunge overhead e potenziali colli di bottiglia, ma ogni livello offre anche opportunità di ottimizzazione. Aleph, se non hai familiarità, è una piattaforma per investigazioni che ge...
------------------------------------------------------------


## 💾 Cella 7: Aggiunta di Esempi Custom (Opzionale ma Raccomandato)

Qui puoi aggiungere coppie domanda-risposta **scritte a mano** per insegnare comportamenti specifici. Questi esempi **manuali** pesano molto nel training e migliorano significativamente la qualità.

In [8]:
# ============================================================
# CELLA 7: ESEMPI CUSTOM (personalizza qui!)
# ============================================================
# Aggiungi qui coppie Q&A specifiche per il tuo caso d'uso.
# Questi esempi 'manuali' sono fondamentali per guidare
# il comportamento del modello su casistiche specifiche.
#
# CONSIGLIO PRO: crea 20-50 esempi di alta qualità, sono
# più efficaci di 1000 esempi generati automaticamente.

custom_examples_raw = [
    # Formato: (domanda_utente, risposta_assistente)
    # ↓ MODIFICA QUESTI CON I TUOI ESEMPI SPECIFICI ↓
    (
        "Chi sei e di cosa ti occupi?",
        "Sono un assistente AI specializzato nell'analisi e nell'elaborazione "
        "dei documenti aziendali. Posso aiutarti a comprendere, riassumere e "
        "analizzare qualsiasi documento tu mi fornisca."
    ),
    (
        "Come posso usarti al meglio?",
        "Puoi chiedermi di: riassumere documenti, spiegare concetti complessi, "
        "rispondere a domande specifiche sul contenuto, confrontare informazioni "
        "tra più documenti, o estrarre dati chiave da testi strutturati."
    ),
    # ↑ AGGIUNGI ALTRI ESEMPI QUI ↑
    # (
    #     "Tua domanda personalizzata?",
    #     "La tua risposta perfetta qui."
    # ),
]

# Converti le tuple nel formato di training corretto
custom_examples = [
    {
        "text": (
            f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
            f"<|user|>\n{question}<|end|>\n"
            f"<|assistant|>\n{answer}<|end|>"
        )
    }
    for question, answer in custom_examples_raw
]

# Combina esempi automatici + manuali
# Gli esempi manuali vengono ripetuti 5 volte per dargli più peso
# (tecnica comune nel fine-tuning per enfatizzare comportamenti chiave)
REPEAT_CUSTOM = 5  # Quante volte ripetere gli esempi manuali

all_examples = training_examples + (custom_examples * REPEAT_CUSTOM)

# Converti in Dataset HuggingFace
# Dataset.from_list() accetta una lista di dizionari
dataset = Dataset.from_list(all_examples)

# Shuffling: mescola il dataset per evitare bias nell'ordine
# seed=42 garantisce riproducibilità (stesso shuffle ogni volta)
dataset = dataset.shuffle(seed=42)

print(f"✅ Dataset finale: {len(dataset)} esempi totali")
print(f"   - Da documenti: {len(training_examples)}")
print(f"   - Custom (x{REPEAT_CUSTOM}): {len(custom_examples) * REPEAT_CUSTOM}")
print(f"\n📊 Struttura dataset: {dataset}")

✅ Dataset finale: 411 esempi totali
   - Da documenti: 401
   - Custom (x5): 10

📊 Struttura dataset: Dataset({
    features: ['text'],
    num_rows: 411
})


In [9]:
# ============================================================
# Diagnostic Cell: Ensure bitsandbytes is importable
# ============================================================
# Explicitly import bitsandbytes to ensure its CUDA kernels are loaded.
# This can sometimes resolve import errors when `transformers` tries to use it.
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes importato correttamente (versione: {bnb.__version__}).")
except ImportError as e:
    print(f"❌ Errore durante l'importazione di bitsandbytes: {e}")
    print("   Ciò potrebbe indicare un problema con l'installazione o la compatibilità CUDA.")
    print("   Si consiglia di riavviare il runtime di Colab (Runtime -> Restart runtime) e rieseguire le celle.")
    raise

✅ bitsandbytes importato correttamente (versione: 0.45.3).


## 🤖 Cella 8: Caricamento Modello con Quantizzazione 4-bit

**QLoRA = Quantized LoRA**: carica il modello in 4-bit per ridurre drasticamente l'uso di VRAM.

- **Phi-3.5-mini a 16-bit**: ~7 GB VRAM ❌ (troppo per T4)
- **Phi-3.5-mini a 4-bit**: ~2.5 GB VRAM ✅ (perfetto per T4)

Questa cella descrive il trucco ingegneristico più importante quando si lavora con modelli di linguaggio (LLM) su hardware commerciale o schede video gratuite (come la NVIDIA T4 presente su Colab, Kaggle o SageMaker Studio Lab).

Senza questa cella, il tuo codice andrebbe immediatamente in Out Of Memory (OOM), bloccando l'esecuzione. Vediamo nel dettaglio cosa succede dietro le quinte.

1. Il problema: Il peso della VRAM (Video RAM)I modelli come Phi-3.5-mini (che ha circa 3.8 miliardi di parametri) sono composti da gigantesche matrici di numeri chiamati pesi o parametri. Ogni peso è memorizzato come un numero decimale.A 16-bit (Precisione nativa / FP16): Ogni singolo parametro occupa 16 bit (ovvero 2 Byte) di memoria sulla scheda video.$$\text{3.8 miliardi di parametri} \times 2 \text{ Byte} \approx 7.6 \text{ GB di VRAM}$$Questo è solo il peso del modello spento. Quando inizi l'addestramento (fine-tuning), devi calcolare i gradienti e memorizzare gli stati dell'ottimizzatore, il che raddoppia o triplica la richiesta di memoria, superando ampiamente i 15-16 GB disponibili su una GPU T4 standard. Il risultato è il crash del sistema (❌ troppo per T4).

2. La soluzione: Cos'è la Quantizzazione a 4-bit?La quantizzazione è un processo di compressione. Immagina di dover fotografare un paesaggio usando una tavolozza di milioni di sfumature di colori (16-bit) e di doverla ridurre a una tavolozza di soli 16 colori selezionati (4-bit, poiché $2^4 = 16$).Nel contesto degli LLM, la quantizzazione a 4-bit prende i numeri decimali ad alta precisione (es. 0.1234567) e li mappa ("arrotonda") su una scala molto più stretta di soli 16 valori discreti possibili.A 4-bit (NF4 / INT4): Ogni parametro ora occupa solo 4 bit (ovvero 0.5 Byte).$$\text{3.8 miliardi di parametri} \times 0.5 \text{ Byte} \approx 1.9 - 2.5 \text{ GB di VRAM}$$Il vantaggio: Il modello occupa pochissimo spazio (✅ perfetto per T4), lasciando quasi tutta la memoria residua della GPU libera per gestire i dati del tuo fine-tuning.

3. Cos'è QLoRA (Quantized LoRA)?Se riduciamo un modello a 4-bit, i suoi pesi diventano "congelati" e non possono essere modificati direttamente durante l'addestramento, perché l'arrotondamento è troppo drastico per permettere micro-regolazioni matematiche. Qui entra in gioco QLoRA.La tecnica funziona così:Congela il modello base a 4-bit: Phi-3.5-mini viene caricato in modalità super-compressa a 4-bit. Nessuno dei suoi parametri originali verrà toccato o modificato.Inietta gli "Adattatori" (LoRA) a 16-bit: Vengono agganciati al modello dei piccolissimi strati aggiuntivi (chiamati Adapter o matrici di basso rango) ad alta precisione (16-bit). Questi adattatori pesano pochissimo (spesso meno dell'1% del modello totale, qualche decina di Megabyte).Addestra solo gli Adattatori: Durante il fine-tuning, la scheda video aggiornerà solo i pesi degli adattatori a 16-bit. Il modello a 4-bit farà solo da "cervello di base" immobile.

In sintesi, perché questa cella è fondamentale?Consente la democratizzazione dell'IA: permette a te (e a chiunque utilizzi hardware free o economico) di prendere un modello modernissimo come Phi-3.5-mini e addestrarlo sui tuoi documenti personali, ottenendo prestazioni professionali ma spendendo zero in infrastrutture cloud.

In [10]:
# ============================================================
# CELLA 8: CARICAMENTO MODELLO CON QUANTIZZAZIONE 4-BIT
# ============================================================

# Nome del modello su HuggingFace Hub
# Alternativa più piccola: "microsoft/phi-2" (2.7B parametri)
# Alternativa italiana: "swap-uniba/LLaMAntino-3-ANITA-8B-Inst-DPO-ITA"
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

print(f"📥 Caricamento modello: {MODEL_NAME}")
print("   (Prima esecuzione: scarica ~2.5GB da HuggingFace, attendi...)")

# ---- Configurazione Quantizzazione BitsAndBytes (4-bit) ----
bnb_config = BitsAndBytesConfig(

    # load_in_4bit: carica i pesi del modello in formato 4-bit NF4
    # Riduce la VRAM da ~7GB a ~2.5GB (risparmio del 64%!)
    load_in_4bit=True,

    # bnb_4bit_quant_type: tipo di quantizzazione
    # "nf4" = NormalFloat4, ottimizzato per pesi pre-addestrati
    # È superiore a INT4 standard per modelli linguistici
    bnb_4bit_quant_type="nf4",

    # bnb_4bit_compute_dtype: dtype usato per i CALCOLI (non lo storage)
    # torch.float16 = half precision per calcoli veloci su GPU
    # Alternativa: torch.bfloat16 (più stabile su A100, peggio su T4)
    bnb_4bit_compute_dtype=torch.float16,

    # bnb_4bit_use_double_quant: quantizza anche le costanti di quantizzazione
    # Risparmia ulteriore VRAM (~0.4 bit/parametro in meno)
    # Leggera perdita di precisione, generalmente trascurabile
    bnb_4bit_use_double_quant=True,
)

# ---- Carica il Tokenizer ----
# Il tokenizer converte il testo in ID numerici comprensibili dal modello
# trust_remote_code=True: necessario per tokenizer personalizzati (come Phi-3)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right"   # Padding a destra per causal LM (standard)
)

# Aggiungi il padding token se mancante
# Phi-3 usa il token EOS anche come PAD (comune nei modelli istruzione)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("   Impostato pad_token = eos_token")

# ---- Carica il Modello ----
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,

    # Applica la configurazione 4-bit definita sopra
    quantization_config=bnb_config,

    # device_map="auto": HuggingFace decide automaticamente
    # dove mettere ogni layer (GPU, CPU, RAM) in base alla VRAM disponibile
    device_map="auto",

    # trust_remote_code: permette di eseguire codice custom del modello
    trust_remote_code=True,

    # attn_implementation: tipo di attention
    # "eager" = standard (compatibile ovunque)
    # Alternativa: "flash_attention_2" (più veloce ma richiede installazione extra)
    attn_implementation="eager",
)

# Prepara il modello per il training con kbit
# Questa funzione:
# 1. Abilita il gradient checkpointing (risparmia VRAM durante backprop)
# 2. Imposta i layer per usare float32 nelle operazioni critiche
# 3. Attiva il training sugli embedding layer
model.config.use_cache = False  # Disabilita KV cache (non serve durante training)
model.config.pretraining_tp = 1  # Tensor parallelism = 1 (singola GPU)

print(f"\n✅ Modello caricato!")
print(f"   Parametri totali: {sum(p.numel() for p in model.parameters()):,}")

# Mostra quanta VRAM stiamo usando
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"   VRAM utilizzata dopo caricamento: {vram_used:.2f} GB")

📥 Caricamento modello: microsoft/Phi-3.5-mini-instruct
   (Prima esecuzione: scarica ~2.5GB da HuggingFace, attendi...)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]


✅ Modello caricato!
   Parametri totali: 2,009,140,224
   VRAM utilizzata dopo caricamento: 2.26 GB


## 🎯 Cella 9: Configurazione LoRA (Low-Rank Adaptation)

**Come funziona LoRA?**

Invece di aggiornare tutti i parametri del modello (3.8 miliardi!), LoRA inserisce piccole **matrici di adattamento** (A e B) in punti strategici. Solo queste matrici vengono aggiornate durante il training.

```
Original: W (3.8B parametri) → non si aggiorna
LoRA:     W + A×B (pochi milioni) → si aggiornano
```

Questa cella descrive il cuore pulsante dell'efficienza nel Deep Learning moderno. Se la quantizzazione a 4-bit (vista nella cella precedente) serve a ridurre lo spazio occupato dal modello in memoria, LoRA (Low-Rank Adaptation) è la tecnica che serve a ridurre lo sforzo matematico e il tempo necessari per addestrarlo.

Senza LoRA, fare il fine-tuning di un modello da 3.8 miliardi di parametri richiederebbe supercomputer industriali. Vediamo nel dettaglio come funziona questa magia matematica.

1. Il metodo classico (Full Fine-Tuning) vs LoRAIn un fine-tuning tradizionale, quando l'algoritmo impara dai tuoi documenti, deve calcolare e aggiornare i valori di tutti i 3.8 miliardi di parametri (pesi) del modello.Ogni peso cambia di una piccola frazione ($\Delta W$, ovvero "Delta Weight").Questo richiede una quantità immane di memoria per tracciare i calcoli (i gradienti) di ogni singolo parametro. Per una GPU T4, questo approccio è matematicamente impossibile.LoRA cambia completamente paradigma: congela totalmente la gigantesca matrice originale dei pesi ($W$) e dice: "Invece di calcolare la gigantesca matrice di cambiamento $\Delta W$, la scompongo in due matrici minuscole molto più semplici".

2. Il trucco matematico: La scomposizione in matrici A e BImmagina di avere una matrice quadrata enorme di dimensioni $4000 \times 4000$. Questa matrice contiene $16.000.000$ di parametri da aggiornare.LoRA applica un concetto di algebra lineare chiamato riduzione del rango (Low-Rank). Invece di calcolare quella matrice enorme, inserisce due matrici sottili una accanto all'altra, chiamate A e B:La matrice A ha dimensioni $4000 \times r$La matrice B ha dimensioni $r \times 4000$Dove $r$ (Rank) è un numero piccolissimo scelto da te, solitamente 8 o 16.Se moltiplichiamo tra loro la matrice A e la matrice B ($A \times B$), il risultato matematico finale avrà comunque le dimensioni della matrice originaria ($4000 \times 4000$), ma per addestrarle dobbiamo calcolare solo:$$(4000 \times 8) + (8 \times 4000) = 32.000 + 32.000 = \mathbf{64.000 \text{ parametri}}$$Siamo passati dal dover calcolare 16 milioni di variazioni a solone 64 mila. Il risparmio computazionale è superiore al 99%.

3. Cosa succede durante il codice W + A×BQuando il tuo modello Phi-3.5 riceve in input la domanda dell'utente durante il training:Il testo passa attraverso la matrice originale W (che è congelata a 4-bit e non impara nulla, fa solo da "conoscenza generale").In parallelo, il testo passa attraverso le piccole matrici A e B (che sono attive, a 16-bit, e stanno imparando i dettagli dei tuoi documenti).I risultati dei due percorsi vengono sommati: W + (A × B)

.I vantaggi pratici di questa cellaUso della VRAM ridicolo: Poiché aggiorni solo lo 0.1% o l'1% dei parametri totali del modello, i requisiti hardware crollano, permettendoti di fare l'addestramento su una singola GPU gratuita.Nessun "Catastrophic Forgetting": Poiché il modello base ($W$) è congelato, Phi-3.5 non dimenticherà come si parla in italiano o come si ragiona logicamente; apprenderà semplicemente le nuove informazioni dei tuoi documenti tramite gli adattatori.Portabilità (File leggeri): Al termine del training, non avrai un nuovo modello da 7 GB da salvare. Avrai solo un file piccolissimo (chiamato Adapter) di circa 20-50 Megabyte che contiene le matrici A e B, pronto per essere applicato sopra il modello Phi-3.5 originale in qualsiasi momento.

In [11]:
# ============================================================
# CELLA 9: CONFIGURAZIONE LORA
# ============================================================

lora_config = LoraConfig(

    # r (rank): dimensione delle matrici LoRA A e B
    # → Più alto = più parametri da addestrare = modello più espressivo
    #   MA anche più VRAM e rischio di overfitting
    # → Valori tipici: 8, 16, 32, 64
    # → r=16: buon compromesso qualità/efficienza per T4
    r=16,

    # lora_alpha: fattore di scaling degli aggiornamenti LoRA
    # → La formula effettiva è: W' = W + (alpha/r) * A×B
    # → Convenzione comune: alpha = 2 * r (qui: 32)
    # → Alpha alto = aggiornamenti più forti (impara più in fretta, ma instabile)
    lora_alpha=32,

    # target_modules: QUALI layer del modello vengono modificati con LoRA
    # → Per Phi-3: questi sono i layer di attention e MLP
    # → q_proj/k_proj/v_proj/o_proj = Query, Key, Value, Output dell'attention
    # → gate_proj/up_proj/down_proj = layer MLP (feed-forward)
    # → Più moduli = più parametri = più espressività (ma più VRAM)
    target_modules=[
        "q_proj",    # Proiezione Query nell'attention
        "k_proj",    # Proiezione Key nell'attention
        "v_proj",    # Proiezione Value nell'attention
        "o_proj",    # Proiezione Output dell'attention
        "gate_proj", # Gate del MLP (SwiGLU activation)
        "up_proj",   # Proiezione Up del MLP
        "down_proj", # Proiezione Down del MLP
    ],

    # lora_dropout: dropout applicato agli adattatori LoRA
    # → Regolarizzazione per prevenire overfitting
    # → 0.05 = 5% dei neuroni vengono "spenti" casualmente durante training
    # → Dataset piccoli: usa 0.1; dataset grandi: usa 0.0
    lora_dropout=0.05,

    # bias: se addestrare anche i bias dei layer
    # → "none": non addestrare i bias (standard, meno parametri)
    # → "all": addestra tutti i bias (raramente necessario)
    # → "lora_only": addestra solo i bias degli adapter LoRA
    bias="none",

    # task_type: tipo di task per il quale configuriamo LoRA
    # → CAUSAL_LM = Language Model causale (predice il prossimo token)
    # → Altri tipi: SEQ_CLS (classificazione), SEQ_2_SEQ_LM (T5)
    task_type=TaskType.CAUSAL_LM,
)

# Applica LoRA al modello
# get_peft_model 'inserisce' le matrici LoRA nei layer target
# e congela tutti gli altri parametri (requires_grad=False)
model = get_peft_model(model, lora_config)

# ---- Report parametri ----
# Mostra quanti parametri verranno effettivamente aggiornati
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ LoRA configurato e applicato!")
print(f"\n📊 Statistiche parametri:")
print(f"   Parametri totali:      {total_params:>15,}")
print(f"   Parametri addestrabili: {trainable_params:>15,}")
print(f"   Percentuale training:   {100 * trainable_params / total_params:.2f}%")
print(f"\n💡 Solo il {100 * trainable_params / total_params:.2f}% dei parametri")
print(f"   viene aggiornato — questo è il potere di LoRA!")

✅ LoRA configurato e applicato!

📊 Statistiche parametri:
   Parametri totali:        2,018,053,120
   Parametri addestrabili:       8,912,896
   Percentuale training:   0.44%

💡 Solo il 0.44% dei parametri
   viene aggiornato — questo è il potere di LoRA!


## 🏋️ Cella 10: Configurazione e Avvio del Training

Qui definiamo tutti gli **iperparametri** del training. Questi sono i valori più critici da capire e personalizzare.

Questa riga di codice, trainer.train(), è il momento in cui la teoria finisce e inizia l'azione. Fino ad ora hai solo preparato gli ingredienti (il modello Phi-3.5 a 4-bit, le matrici LoRA vuote e il tuo dataset di documenti formattato in ChatML); invocando questa funzione, accendi ufficialmente il motore di addestramento.

Ecco, nel dettaglio e "sotto il cofano", cosa fa la libreria di Hugging Face (Transformers) quando esegui questo comando.

Il Ciclo del Training (Cosa succede alla GPU)
Il comando trainer.train() avvia un ciclo iterativo (chiamato Training Loop) che si ripete migliaia di volte. Per ogni gruppo di domande e risposte (chiamato Batch) estratto dai tuoi documenti, la GPU esegue tre passaggi fondamentali:

1. Il Forward Pass (Il tentativo del modello)
Il trainer prende una domanda dal tuo dataset (es. <|user|>\nQuanti giorni di preavviso servono per le ferie?<|end|>\n<|assistant|>\n) e la dà in pasto al modello.
Il modello, usando la sua conoscenza base combinata con lo stato attuale delle matrici LoRA, prova a indovinare la risposta parola dopo parola.

2. Calcolo della Loss (Il giudizio)
Il trainer confronta la risposta generata dal modello con la risposta reale presente nel tuo dataset.
Viene calcolato un valore matematico chiamato Loss (Perdita). Più la risposta del modello è diversa da quella corretta, più la Loss è alta. L'obiettivo di tutto il training è portare la Loss il più vicino possibile a zero.

3. Il Backward Pass e l'Ottimizzazione (L'apprendimento)
Tramite un algoritmo matematico (la Backpropagation), l'errore commesso viene proiettato all'indietro.
Il trainer calcola di quanto modificare i parametri delle sole matrici LoRA (A e B) per fare in modo che, la prossima volta che vedrà una domanda simile, l'errore sia minore. I pesi originali di Phi-3.5 a 4-bit, come sappiamo, restano congelati.

Da cosa dipendono i tempi d'attesa?
I commenti nel tuo codice sono precisi: l'attesa può variare da una decina di minuti a diverse ore. Questo dipende da tre fattori logistici:

A. Dimensione del Dataset
Più documenti e coppie di domande/risposte hai creato nella Cella 6, più passi (step) la GPU dovrà compiere per completare il giro. Se hai 100 esempi ci vorranno pochissimi minuti; se ne hai 50.000, l'attesa sarà molto più lunga.

B. Numero di Epoche (Epochs)
Un'Epoca rappresenta un giro completo di addestramento su tutto il tuo dataset.
Se imposti num_train_epochs = 3, significa che il modello leggerà e studierà i tuoi documenti per ben 3 volte consecutive. Più epoche significano un modello potenzialmente più preciso, ma moltiplicano linearmente il tempo richiesto.

C. La GPU Disponibile
La velocità con cui vengono eseguiti i calcoli matriciali descritti sopra dipende interamente dal silicio su cui stai girando:

NVIDIA T4 (Free su Colab/Kaggle): È una scheda eccellente per iniziare e per l'addestramento con QLoRA, ma ha una velocità di calcolo moderata.

NVIDIA A10G / L4 (Cloud a pagamento): Possono essere da 2 a 4 volte più veloci di una T4.

NVIDIA A100 / H100 (Enterprise): Abbattono i tempi di ore in pochissimi minuti, ma hanno costi orari elevati.

Cosa vedrai apparire sullo schermo?
Mentre trainer.train() è in esecuzione, vedrai comparire una tabella o una barra di avanzamento che si aggiorna periodicamente. I parametri fondamentali da monitorare sono due:

Training Loss: È il valore dell'errore. Deve scendere progressivamente con il passare dei minuti (es. da 2.5 a 0.8 o meno). Se vedi che scende, significa che il modello sta effettivamente imparando.

Gradual Progress / ETA: Ti mostrerà il tempo stimato rimanente per la fine del processo.

In [16]:
from trl import SFTTrainer, SFTConfig

# ============================================================
# CELLA 10: CONFIGURAZIONE TRAINING E AVVIO
# ============================================================

# Cartella dove salvare checkpoint e modello finale
OUTPUT_DIR = "/content/finetuned_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- TrainingArguments: tutti gli iperparametri del training ----
training_args = TrainingArguments(

    # Cartella di output per checkpoint e log
    output_dir=OUTPUT_DIR,

    # num_train_epochs: quante volte il modello vede l'intero dataset
    # → 1 epoch: veloce ma superficiale
    # → 3-5 epoch: standard per fine-tuning (buon bilanciamento)
    # → >10 epoch: rischio overfitting su dataset piccoli
    num_train_epochs=3,

    # per_device_train_batch_size: quanti esempi vengono processati insieme
    # → Batch più grande = training più stabile MA più VRAM
    # → T4 (15GB): usa batch_size=2 (sicuro) o 4 (rischiando OOM)
    # → Se ottieni OOM (Out of Memory): riduci a 1
    per_device_train_batch_size=2,

    # gradient_accumulation_steps: accumula i gradienti prima di aggiornare
    # → Batch EFFETTIVO = batch_size × gradient_accumulation_steps
    # → Qui: 2 × 8 = 16 (batch effettivo di 16 esempi)
    # → Simula un batch grande senza usare più VRAM!
    gradient_accumulation_steps=8,

    # gradient_checkpointing: ricalcola i gradient invece di salvarli
    # → Risparmia ~30-40% VRAM durante il backprop
    # → Training ~20% più lento (trade-off valido su Colab)
    gradient_checkpointing=True,
    # Aggiungi questa linea per risolvere il warning di PyTorch 2.9
    gradient_checkpointing_kwargs={'use_reentrant': False},

    # learning_rate: velocità di apprendimento
    # → Troppo alto: il modello diverge (loss aumenta)
    # → Troppo basso: training lentissimo
    # → 2e-4 (0.0002): ottimale per LoRA con scheduler cosine
    learning_rate=2e-4,

    # lr_scheduler_type: come il learning rate cambia durante il training
    # → "cosine": parte alto, scende a zero come una curva coseno
    #   Molto usato per LLM (adattamento graduale)
    # → Alternative: "linear", "constant", "polynomial"
    lr_scheduler_type="cosine",

    # warmup_ratio: % degli step iniziali con LR che cresce da 0 al valore target
    # → 0.03 = 3% degli step totali sono di 'warmup'
    # → Previene instabilità all'inizio del training
    warmup_ratio=0.03,

    # max_grad_norm: gradient clipping — taglia i gradienti troppo grandi
    # → Previene 'gradient explosion' che causa instabilità
    # → 0.3 è ottimale per LoRA (più conservativo del default 1.0)
    max_grad_norm=0.3,

    # weight_decay: L2 regularization per prevenire overfitting
    # → Penalizza pesi troppo grandi
    # → 0.001: valore leggero, standard per fine-tuning
    weight_decay=0.001,

    # fp16: usa floating point a 16-bit (half precision)
    # → Dimezza la VRAM per i calcoli del training
    # → Funziona bene su T4 (Volta/Turing architecture)
    # → Su A100 usa bf16=True invece (più stabile)
    fp16=True,
    bf16=False,

    # logging_steps: ogni quanti step mostrare le metriche (loss, lr)
    logging_steps=10,

    # save_strategy: quando salvare i checkpoint
    # → "epoch": salva alla fine di ogni epoch
    # → "steps": salva ogni N step (specificare save_steps)
    save_strategy="epoch",

    # save_total_limit: mantieni solo gli ultimi N checkpoint
    # → Evita di riempire il disco di Colab
    save_total_limit=2,

    # optim: ottimizzatore
    # → "paged_adamw_32bit": versione paginata di AdamW
    #   Gestisce la memoria in modo più efficiente su GPU
    #   'Paged' = usa memoria CPU come fallback quando la GPU è piena
    optim="paged_adamw_32bit",

    # report_to: dove mandare i log di training
    # → "none": disabilita (non serve WandB o TensorBoard)
    # → "wandb": manda a Weights & Biases (ottimo per monitoraggio)
    report_to="none",

    # group_by_length: raggruppa esempi di lunghezza simile nello stesso batch
    # → Riduce il padding (token vuoti) → training più efficiente
    group_by_length=True,
)

# ---- SFTTrainer: Supervised Fine-Tuning Trainer ----
# SFTTrainer è una versione specializzata di Trainer per instruction tuning
# Gestisce automaticamente:
# - La tokenizzazione del dataset
# - Il packing degli esempi (opzionale)
# - Il masking delle label (per non addestrare sulla parte 'user')

# Per la versione 0.11.1 di trl, i parametri vanno passati direttamente a SFTTrainer
# SFTConfig è per versioni successive.

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    # dataset_text_field: quale campo del dataset contiene il testo
    dataset_text_field="text",
    # max_seq_length: lunghezza massima delle sequenze in token
    max_seq_length=1024,
    # packing: combina più esempi corti in una singola sequenza
    packing=False,
)

print("✅ Trainer configurato! Avvio del training...")
print("\n" + "=" * 60)
print("🚀 INIZIO TRAINING")
print("=" * 60)
print(f"   Dataset: {len(dataset)} esempi")
print(f"   Epoche:  {training_args.num_train_epochs}")
print(f"   Batch effettivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   LR: {training_args.learning_rate}")
print("=" * 60 + "\n")

# Avvia il training!
# Questo richiede da 10 minuti a qualche ora dipendendo da:
# - Dimensione del dataset
# - Numero di epoche
# - GPU disponibile
trainer.train()

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETATO!")
print("=" * 60)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/411 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


✅ Trainer configurato! Avvio del training...

🚀 INIZIO TRAINING
   Dataset: 411 esempi
   Epoche:  3
   Batch effettivo: 16
   LR: 0.0002



Step,Training Loss
10,2.092000
20,1.844500
30,1.720100
40,1.735900
50,1.614000
60,1.665600
70,1.637600



✅ TRAINING COMPLETATO!


In [17]:
# ============================================================
# CELLA 11: SALVATAGGIO DEL MODELLO
# ============================================================

FINAL_MODEL_DIR = "/content/final_model"
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ---- Opzione 1: Salva solo gli Adapter LoRA (RACCOMANDATO) ----
# Salva solo le matrici LoRA addestrate (~30-100 MB)
# Per usarle serve il modello base originale
ADAPTER_DIR = os.path.join(FINAL_MODEL_DIR, "lora_adapters")
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ Adapter LoRA salvati in: {ADAPTER_DIR}")

# ---- Opzione 2: Salva il modello MERGED (base + LoRA fusi) ----
# Produce un modello standalone completo (più grande, ~6-8 GB)
# Utile per deployment o per caricare senza PEFT
# NOTA: richiede più VRAM, usa float16 per ridurre le dimensioni

print("\n🔄 Merging del modello base con gli adapter LoRA...")
from peft import PeftModel

# Ricarica il modello base in float16 (non 4-bit, per il merge)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Carica gli adapter LoRA sopra il modello base
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

# merge_and_unload(): fonde matematicamente A×B nel peso W
# → Risultato: W' = W + (alpha/r) * A×B
# → Il modello risultante NON ha più bisogno di PEFT
merged_model = merged_model.merge_and_unload()

MERGED_DIR = os.path.join(FINAL_MODEL_DIR, "merged_model")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print(f"✅ Modello merged salvato in: {MERGED_DIR}")

# ---- Mostra le dimensioni dei file ----
import shutil
adapter_size = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob('*') if f.is_file()) / 1e6
merged_size  = sum(f.stat().st_size for f in Path(MERGED_DIR).rglob('*') if f.is_file()) / 1e6
print(f"\n📦 Dimensioni:")
print(f"   Adapter LoRA: {adapter_size:.1f} MB")
print(f"   Merged model: {merged_size:.1f} MB")

✅ Adapter LoRA salvati in: /content/final_model/lora_adapters

🔄 Merging del modello base con gli adapter LoRA...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Modello merged salvato in: /content/final_model/merged_model

📦 Dimensioni:
   Adapter LoRA: 38.0 MB
   Merged model: 7644.6 MB


## 🧪 Cella 12: Test del Modello Fine-Tuned

Ora testiamo il modello con alcune domande per verificare che abbia appreso il contenuto dei documenti.

In [18]:
# ============================================================
# CELLA 12: TEST E INFERENCE DEL MODELLO
# ============================================================

def generate_response(model, tokenizer, user_message: str,
                       system_prompt: str = None,
                       max_new_tokens: int = 512,
                       temperature: float = 0.7,
                       top_p: float = 0.9) -> str:
    """
    Genera una risposta dal modello fine-tuned.

    Parametri di generazione:

    max_new_tokens: quanti token generare al massimo
        → 256: risposte brevi
        → 512: risposte medie (default)
        → 1024: risposte lunghe

    temperature: 'creatività' del modello
        → 0.0: completamente deterministico (sempre la stessa risposta)
        → 0.7: buon bilanciamento (creativo ma coerente)
        → 1.0: molto creativo (può diventare incoerente)
        → >1.5: quasi casuale

    top_p (nucleus sampling):
        → Considera solo i token che sommano al top_p% di probabilità
        → 0.9: considera i token più probabili fino al 90% cumulativo
        → 1.0: considera tutti i token (no filtro)
        → 0.5: molto conservativo, solo i token più probabili
    """

    if system_prompt is None:
        system_prompt = SYSTEM_PROMPT

    # Costruisci il prompt nel formato ChatML di Phi-3
    prompt = (
        f"<|system|>\n{system_prompt}<|end|>\n"
        f"<|user|>\n{user_message}<|end|>\n"
        f"<|assistant|>\n"
    )

    # Tokenizza il prompt
    # return_tensors="pt": restituisce tensori PyTorch (per GPU)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_length = inputs["input_ids"].shape[1]  # Lunghezza del prompt

    # Genera la risposta
    with torch.no_grad():  # Disabilita gradient (non serve per inference)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,

            # do_sample=True: usa sampling stocastico (temperature/top_p attivi)
            # do_sample=False: greedy decoding (sempre il token più probabile)
            do_sample=True,

            # repetition_penalty: penalizza la ripetizione di token già generati
            # → 1.0: nessuna penalità
            # → 1.1-1.3: leggera penalità (evita loop)
            repetition_penalty=1.1,

            # eos_token_id: token che segnala fine della generazione
            eos_token_id=tokenizer.eos_token_id,

            # pad_token_id: token di padding (uguale a EOS per Phi-3)
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decodifica solo i nuovi token (escludi il prompt dall'output)
    new_tokens = outputs[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response.strip()


# ---- Test con domande di esempio ----
# Modifica queste domande con quelle pertinenti ai tuoi documenti!
test_questions = [
    "Chi sei e cosa puoi fare per me?",
    "Riassumi i punti principali dei documenti che hai studiato.",
    # Aggiungi qui domande specifiche sul contenuto dei tuoi documenti:
    # "Quali sono le caratteristiche del prodotto X?",
    # "Spiega il processo descritto nel manuale.",
]

print("🧪 TEST DEL MODELLO FINE-TUNED")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n📌 Domanda {i}: {question}")
    print("-" * 40)

    response = generate_response(
        trainer.model,  # Usa il modello appena addestrato
        tokenizer,
        question,
        max_new_tokens=300,
        temperature=0.7,
    )

    print(f"🤖 Risposta:\n{response}")
    print("=" * 60)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


🧪 TEST DEL MODELLO FINE-TUNED

📌 Domanda 1: Chi sei e cosa puoi fare per me?
----------------------------------------
🤖 Risposta:
Sono Phi-AssistAI, progettato appositamente per aiutarti con l'elaborazione di grandi quantità di testo. Posso analizzare, riassumere, confrontare, verificare fatti o spiegare concetti complessi presenti nei file che mi fornisci. Ecco alcune delle mie principali capacità:

1. Riassunto (Summarization): Puoi chiedermi di ridurre i punti salienti di qualsiasi capitolo o articolo a una versione più breve ed essenziale.
2. Analisi Comparativa (Comparison Analysis): Crea diagrammi comparativi tra due fonti riguardanti lo stesso argomento se ne hai uno specifico in mente.
3. Verifica Factual Accurata (Factual Validation): Controllo la correttezza factuale all'interno del tuo materiale quando è importante il rigore scientifico/giudiziario.
4. Spiegazione Detailista (Detailed Explanations): Analizziamo passaggi tecnici, processi logici o strutture complesse come que

## ☁️ Cella 13: Salvataggio su Google Drive

Salva il modello su Google Drive per non perderlo alla chiusura di Colab!

In [ ]:
# ============================================================
# CELLA 13: BACKUP SU GOOGLE DRIVE (OPZIONALE)
# ============================================================
# Colab cancella tutto alla chiusura della sessione!
# Salva il modello su Drive per conservarlo.

SAVE_TO_DRIVE = True  # Imposta False per saltare

if SAVE_TO_DRIVE:
    from google.colab import drive

    # Monta Google Drive in /content/drive
    # Apparirà una richiesta di autorizzazione
    drive.mount('/content/drive')

    # Cartella di destinazione su Drive
    DRIVE_DEST = "/content/drive/MyDrive/finetuned_models/my_model"
    os.makedirs(DRIVE_DEST, exist_ok=True)

    print("📤 Copia degli adapter su Google Drive...")
    print("   (Solo gli adapter: pochi MB, molto veloce)")

    # Copia solo gli adapter (leggeri) su Drive
    shutil.copytree(
        ADAPTER_DIR,
        os.path.join(DRIVE_DEST, "lora_adapters"),
        dirs_exist_ok=True
    )

    print(f"\n✅ Adapter salvati su Drive: {DRIVE_DEST}/lora_adapters")
    print("\nPer ricaricare il modello in futuro:")
    print(f"   model = PeftModel.from_pretrained(base_model, '{DRIVE_DEST}/lora_adapters')")
else:
    print("ℹ️ Salvataggio su Drive saltato.")

## 📤 Cella 14: Caricamento su HuggingFace Hub (Opzionale)

Carica il modello su HuggingFace per condividerlo o usarlo in produzione.

In [ ]:
# ============================================================
# CELLA 14: UPLOAD SU HUGGINGFACE HUB (OPZIONALE)
# ============================================================
# Carica il modello su HuggingFace per usarlo da qualsiasi luogo.
# Richiede un account HuggingFace gratuito e un token API.

UPLOAD_TO_HF = False  # Imposta True per caricare su HuggingFace

if UPLOAD_TO_HF:
    from huggingface_hub import login

    # Ottieni il tuo token da: https://huggingface.co/settings/tokens
    HF_TOKEN = "hf_xxx..."   # ← SOSTITUISCI CON IL TUO TOKEN
    HF_USERNAME = "tuo_username"  # ← SOSTITUISCI CON IL TUO USERNAME
    REPO_NAME = "phi35-mini-finetuned-ita"  # ← Nome del repository

    login(token=HF_TOKEN)

    # Carica gli adapter LoRA
    # push_to_hub() carica automaticamente su HuggingFace
    trainer.model.push_to_hub(
        f"{HF_USERNAME}/{REPO_NAME}",
        use_auth_token=HF_TOKEN,
        private=True  # True = repo privato; False = pubblico
    )
    tokenizer.push_to_hub(
        f"{HF_USERNAME}/{REPO_NAME}",
        use_auth_token=HF_TOKEN
    )

    print(f"✅ Modello caricato su: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")
else:
    print("ℹ️ Upload su HuggingFace saltato.")
    print("   Per attivarlo: imposta UPLOAD_TO_HF = True e inserisci il tuo token.")

## 🔄 Cella 15: Ricaricamento del Modello (Sessioni Future)

Come ricaricare e usare il modello in una nuova sessione Colab.

In [ ]:
# ============================================================
# CELLA 15: RICARICAMENTO MODELLO (USA IN FUTURO)
# ============================================================
# Questa cella NON va eseguita ora.
# Usala in una NUOVA sessione Colab per ricaricare il modello.

# ---- ISTRUZIONI PER NUOVA SESSIONE ----
# 1. Monta Google Drive: drive.mount('/content/drive')
# 2. Installa le dipendenze (Cella 1)
# 3. Esegui questa cella per ricaricare il modello

def reload_finetuned_model(
    base_model_name: str = "microsoft/Phi-3.5-mini-instruct",
    adapter_path: str = "/content/drive/MyDrive/finetuned_models/my_model/lora_adapters"
):
    """
    Ricarica il modello fine-tuned con gli adapter LoRA salvati.

    Processo:
    1. Carica il modello base in 4-bit (veloce, poca VRAM)
    2. Carica gli adapter LoRA sopra il base
    3. Pronto per l'inference!
    """
    from peft import PeftModel

    print(f"📥 Caricamento modello base: {base_model_name}")

    # Configurazione 4-bit (stessa usata durante il training)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Ricarica tokenizer
    tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Ricarica modello base
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Carica gli adapter LoRA
    print(f"🔌 Caricamento adapter da: {adapter_path}")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()  # Modalità inference (disabilita dropout)

    print("✅ Modello fine-tuned caricato e pronto!")
    return model, tokenizer


# Per usare:
# model, tokenizer = reload_finetuned_model()
# response = generate_response(model, tokenizer, "La tua domanda qui")
# print(response)

print("ℹ️ Questa cella contiene la funzione reload_finetuned_model().")
print("   Usala in una NUOVA sessione Colab per ricaricare il modello.")

---

## 📚 Riepilogo dei Parametri Chiave

| Parametro | Valore | Effetto |
|-----------|--------|------|
| `r` (LoRA rank) | 16 | Capacità adapter: alto=più parametri |
| `lora_alpha` | 32 | Scaling aggiornamenti: alto=impara di più |
| `learning_rate` | 2e-4 | Velocità apprendimento |
| `num_train_epochs` | 3 | Passaggi sul dataset |
| `batch_size` | 2 | Esempi per step GPU |
| `gradient_accumulation` | 8 | Batch effettivo = 2×8=16 |
| `max_seq_length` | 1024 | Max token per esempio |
| `temperature` | 0.7 | Creatività risposte |
| `top_p` | 0.9 | Filtro token improbabili |

## 🔧 Troubleshooting Comune

| Problema | Soluzione |
|----------|----------|
| `CUDA out of memory` | Riduci `batch_size=1`, `max_seq_length=512` |
| Training loss non scende | Aumenta `learning_rate` o `num_epochs` |
| Loss va a NaN | Riduci `learning_rate`, abilita `max_grad_norm` |
| Modello ripete testo | Aumenta `repetition_penalty` a 1.2-1.3 |
| Risposte incoerenti | Riduci `temperature` a 0.3-0.5 |

In [19]:
# ============================================================
# CELLA 16: INSTALLAZIONE LLAMA.CPP (CONVERTITORE → GGUF)
# ============================================================
# Ollama usa il formato GGUF (GPT-Generated Unified Format)
# llama.cpp è lo strumento ufficiale per convertire da HuggingFace → GGUF
#
# Compiliamo llama.cpp direttamente da sorgente su Colab
# perché la versione pip non include sempre convert_hf_to_gguf.py

import os

# Clona il repository llama.cpp
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp

# Installa le dipendenze Python di llama.cpp per la conversione
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

print("✅ llama.cpp installato e pronto per la conversione!")

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 99381, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 99381 (delta 70), reused 31 (delta 31), pack-reused 99252 (from 4)
Receiving objects: 100% (99381/99381), 399.78 MiB | 32.07 MiB/s, done.
Resolving deltas: 100% (70073/70073), done.
Updating files: 100% (2994/2994), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.9 MB/s eta 0:00:00
ERROR: pip's dependency resol

In [21]:
# ============================================================
# CELLA 17: CONVERSIONE DEL MODELLO IN FORMATO GGUF
# ============================================================
# PREREQUISITO: la Cella 11 deve essere già stata eseguita
# Il modello merged deve essere in: /content/final_model/merged_model
#
# TIPI DI QUANTIZZAZIONE GGUF disponibili:
# ┌─────────────┬──────────┬─────────────────────────────────────┐
# │ Tipo        │ Dim.     │ Quando usarlo                       │
# ├─────────────┼──────────┼─────────────────────────────────────┤
# │ f16         │ ~7 GB    │ Massima qualità, RAM abbondante     │
# │ q8_0        │ ~4 GB    │ Ottima qualità, buona velocità      │
# │ q4_k_m      │ ~2.5 GB  │ ✅ CONSIGLIATO: miglior compromesso │
# │ q4_0        │ ~2 GB    │ Veloce, qualità leggermente minore  │
# │ q2_k        │ ~1.5 GB  │ RAM minima, qualità ridotta         │
# └─────────────┴──────────┴─────────────────────────────────────┘

MERGED_DIR  = "/content/final_model/merged_model"
GGUF_DIR    = "/content/final_model/gguf"
GGUF_NAME   = "phi35-mini-finetuned"

# Tipo di quantizzazione — q4_k_m è il migliore per uso desktop
# Cambia in "q8_0" se hai 8+ GB di RAM libera e vuoi più qualità
QUANT_TYPE  = "q4_k_m"

os.makedirs(GGUF_DIR, exist_ok=True)

# ---- Step 1: Converti da HuggingFace → GGUF (formato f16 intermedio) ----
# convert_hf_to_gguf.py legge i pesi HuggingFace e li converte in GGUF
# --outtype f16: prima convertiamo in float16 (passaggio intermedio)
# --outfile: percorso del file GGUF di output

GGUF_F16_PATH = f"{GGUF_DIR}/{GGUF_NAME}-f16.gguf"

print("🔄 Step 1/2: Conversione HuggingFace → GGUF (f16)...")
print("   (Potrebbe richiedere 5-10 minuti...)\n")

!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outtype f16 \
    --outfile {GGUF_F16_PATH}

print(f"\n✅ GGUF f16 creato: {GGUF_F16_PATH}")

# ---- Step 2: Quantizza il GGUF in q4_k_m (molto più piccolo) ----
# llama-quantize è il binario C++ di llama.cpp per la quantizzazione
# Riduce ulteriormente le dimensioni mantenendo buona qualità
# Dobbiamo prima compilare llama.cpp

GGUF_QUANT_PATH = f"{GGUF_DIR}/{GGUF_NAME}-{QUANT_TYPE}.gguf"

print(f"\n🔄 Step 2/2: Compilazione llama.cpp e quantizzazione → {QUANT_TYPE}...")
print("   (Compilazione ~3 minuti, quantizzazione ~2 minuti...)\n")

# Compila llama.cpp (serve per llama-quantize)
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF 2>/dev/null && \
 cmake --build build --config Release -j$(nproc) --target llama-quantize 2>/dev/null

# Esegui la quantizzazione
!cd /content/llama.cpp && ./build/bin/llama-quantize \
    {GGUF_F16_PATH} \
    {GGUF_QUANT_PATH} \
    {QUANT_TYPE.upper()}

# Mostra dimensioni dei file
import shutil
from pathlib import Path

size_f16   = Path(GGUF_F16_PATH).stat().st_size / 1e9
size_quant = Path(GGUF_QUANT_PATH).stat().st_size / 1e9

print(f"\n📦 File GGUF generati:")
print(f"   f16  (intermedio): {size_f16:.2f} GB  → {GGUF_F16_PATH}")
print(f"   {QUANT_TYPE} (finale):   {size_quant:.2f} GB  → {GGUF_QUANT_PATH}")
print(f"\n💡 Il file da scaricare è: {GGUF_NAME}-{QUANT_TYPE}.gguf")

🔄 Step 1/2: Conversione HuggingFace → GGUF (f16)...
   (Potrebbe richiedere 5-10 minuti...)

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Phi3ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_factors_long.weight,  torch.float32 --> F32, shape = {48}
INFO:hf-to-gguf:rope_factors_short.weight, torch.float32 --> F32, shape = {48}
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16

In [22]:
# ============================================================
# CELLA 18: CREAZIONE MODELFILE PER OLLAMA
# ============================================================
# Il Modelfile è il "Dockerfile di Ollama" — dice a Ollama
# come caricare e configurare il modello:
# - Quale file GGUF usare
# - Il system prompt (persona del modello)
# - I parametri di generazione (temperatura, ecc.)
# - Il template di conversazione (formato ChatML per Phi-3)

MODELFILE_PATH = f"{GGUF_DIR}/Modelfile"

# Nome del modello come apparirà in Ollama
OLLAMA_MODEL_NAME = "mio-assistente"

modelfile_content = f'''# Modelfile generato automaticamente per Ollama
# Modello: Phi-3.5-mini fine-tuned con QLoRA

# FROM: percorso al file GGUF (dopo averlo scaricato sul desktop)
# ⚠️  CAMBIA QUESTO PERCORSO con quello reale sul tuo PC!
FROM ./{GGUF_NAME}-{QUANT_TYPE}.gguf

# TEMPLATE: formato di conversazione ChatML usato da Phi-3
# Ollama usa questo template per costruire i prompt correttamente
# {{{{ .System }}}} e {{{{ .Prompt }}}} sono variabili sostituite da Ollama
TEMPLATE """<|system|>
{{{{ .System }}}}<|end|>
<|user|>
{{{{ .Prompt }}}}<|end|>
<|assistant|>
{{{{ .Response }}}}<|end|>
"""

# SYSTEM: prompt di sistema predefinito (personalizza con la tua persona)
SYSTEM """{SYSTEM_PROMPT}"""

# PARAMETER: parametri di generazione
# temperature: creatività (0.0=deterministico, 1.0=creativo)
PARAMETER temperature 0.7

# top_p: nucleus sampling — considera token fino al 90% di probabilità cumulativa
PARAMETER top_p 0.9

# top_k: considera solo i top-k token più probabili ad ogni step
PARAMETER top_k 40

# repeat_penalty: penalizza la ripetizione di parole/frasi
PARAMETER repeat_penalty 1.1

# num_ctx: dimensione della finestra di contesto in token
# 4096 = buon compromesso memoria/contesto per Phi-3.5-mini
PARAMETER num_ctx 4096

# num_predict: numero massimo di token da generare per risposta
# -1 = nessun limite (il modello si ferma da solo con EOS token)
PARAMETER num_predict 512

# stop: token che fermano la generazione
PARAMETER stop "<|end|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|system|>"
'''

with open(MODELFILE_PATH, 'w') as f:
    f.write(modelfile_content)

print(f"✅ Modelfile creato: {MODELFILE_PATH}")
print("\n📄 Contenuto del Modelfile:")
print("-" * 50)
print(modelfile_content)

✅ Modelfile creato: /content/final_model/gguf/Modelfile

📄 Contenuto del Modelfile:
--------------------------------------------------
# Modelfile generato automaticamente per Ollama
# Modello: Phi-3.5-mini fine-tuned con QLoRA

# FROM: percorso al file GGUF (dopo averlo scaricato sul desktop)
# ⚠️  CAMBIA QUESTO PERCORSO con quello reale sul tuo PC!
FROM ./phi35-mini-finetuned-q4_k_m.gguf

# TEMPLATE: formato di conversazione ChatML usato da Phi-3
# Ollama usa questo template per costruire i prompt correttamente
# {{ .System }} e {{ .Prompt }} sono variabili sostituite da Ollama
TEMPLATE """<|system|>
{{ .System }}<|end|>
<|user|>
{{ .Prompt }}<|end|>
<|assistant|>
{{ .Response }}<|end|>
"""

# SYSTEM: prompt di sistema predefinito (personalizza con la tua persona)
SYSTEM """Sei un assistente AI specializzato nell'analisi dei documenti forniti. Rispondi sempre in italiano in modo preciso, dettagliato e professionale. Le tue risposte sono basate esclusivamente sulle informazioni conten

In [26]:
# ============================================================
# CELLA 19: DOWNLOAD FILE E ISTRUZIONI PER IL DESKTOP
# ============================================================

from google.colab import files
import zipfile

print("📥 Preparazione file da scaricare...\n")

# Crea uno zip con GGUF + Modelfile
ZIP_PATH = f"/content/{GGUF_NAME}-ollama.zip"

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Aggiungi il file GGUF quantizzato
    zipf.write(GGUF_QUANT_PATH, f"{GGUF_NAME}-{QUANT_TYPE}.gguf")
    # Aggiungi il Modelfile
    zipf.write(MODELFILE_PATH, "Modelfile")

zip_size = Path(ZIP_PATH).stat().st_size / 1e9
print(f"📦 ZIP creato: {ZIP_PATH} ({zip_size:.2f} GB)")
print("\n⬇️  Avvio download...")

# Scarica il file zip su Colab
files.download(ZIP_PATH)

# ---- Istruzioni per il desktop ----
print("""
╔══════════════════════════════════════════════════════════════╗
║           ISTRUZIONI PER INSTALLARE SU OLLAMA                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. INSTALLA OLLAMA (se non l'hai già):                      ║
║     → Windows/Mac: https://ollama.com/download               ║
║     → Linux: curl -fsSL https://ollama.ai/install.sh | sh   ║
║                                                              ║
║  2. DECOMPRIMI lo zip scaricato in una cartella, es:         ║
║     C:\\ollama-models\\mio-assistente\\                         ║
║                                                              ║
║  3. APRI IL TERMINALE nella cartella decompressa e lancia:   ║
║                                                              ║
║     ollama create mio-assistente -f ./Modelfile              ║
║                                                              ║
║  4. USA IL MODELLO:                                          ║
║                                                              ║
║     # Chat interattiva nel terminale:                        ║
║     ollama run mio-assistente                                ║
║                                                              ║
║     # API REST locale (porta 11434):                         ║
║     curl http://localhost:11434/api/generate -d '{           ║
║       "model": "mio-assistente",                             ║
║       "prompt": "Ciao, chi sei?"                             ║
║     }'                                                       ║
║                                                              ║
║  5. LISTA MODELLI INSTALLATI:                                ║
║     ollama list                                              ║
║                                                              ║
║  6. RIMUOVI IL MODELLO (se vuoi):                            ║
║     ollama rm mio-assistente                                 ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

📥 Preparazione file da scaricare...

📦 ZIP creato: /content/phi35-mini-finetuned-ollama.zip (2.36 GB)

⬇️  Avvio download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


╔══════════════════════════════════════════════════════════════╗
║           ISTRUZIONI PER INSTALLARE SU OLLAMA                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. INSTALLA OLLAMA (se non l'hai già):                      ║
║     → Windows/Mac: https://ollama.com/download               ║
║     → Linux: curl -fsSL https://ollama.ai/install.sh | sh   ║
║                                                              ║
║  2. DECOMPRIMI lo zip scaricato in una cartella, es:         ║
║     C:\ollama-models\mio-assistente\                         ║
║                                                              ║
║  3. APRI IL TERMINALE nella cartella decompressa e lancia:   ║
║                                                              ║
║     ollama create mio-assistente -f ./Modelfile              ║
║                                                              ║
║  4. USA IL MODELLO:    